# Hale & Nicholson (1925) — The Law of Sun-Spot Polarity: Implementation
# 흑점 극성 법칙: 구현

**Paper / 논문**: G.E. Hale & S.B. Nicholson, "The Law of Sun-Spot Polarity," *ApJ*, 62, 270–300, 1925.

이 노트북은 Hale의 극성 법칙의 핵심 물리학과 데이터를 시각화합니다:

1. **Part 1**: Butterfly Diagram + Polarity — Fig. 18 재현 / Reproducing Fig. 18
2. **Part 2**: 22년 자기 주기 시각화 — 극성 반전 패턴 / 22-year magnetic cycle visualization
3. **Part 3**: 분류 통계 — Tables I–V 재현 / Classification statistics
4. **Part 4**: 쌍극 흑점군의 자기 구조 모델 / Magnetic structure model of bipolar groups

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, Ellipse
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    'figure.figsize': (14, 7),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## Part 1: Synthetic Butterfly Diagram with Polarity — Reproducing Fig. 18
## 파트 1: 합성 나비 다이어그램 + 극성 — Fig. 18 재현

Hale의 Fig. 18을 현대적으로 재현합니다. 나비 다이어그램(위도 vs. 시간)에 극성 정보를 추가하여
22년 자기 주기를 시각화합니다.

Reproducing Hale's Fig. 18 with modern visualization. Adding polarity information to the
butterfly diagram (latitude vs. time) to visualize the 22-year magnetic cycle.

In [ ]:
def generate_butterfly_with_polarity(n_cycles: int = 4, spots_per_cycle: int = 500,
                                     seed: int = 42) -> dict:
    """Generate synthetic butterfly diagram data with Hale polarity law.

    Args:
        n_cycles: Number of 11-year half-cycles to simulate.
        spots_per_cycle: Number of spots per half-cycle.
        seed: Random seed.

    Returns:
        Dictionary with 'time', 'latitude', 'polarity_preceding' arrays.
    """
    rng = np.random.default_rng(seed)
    cycle_length = 11.0  # years
    all_time, all_lat, all_pol = [], [], []

    for i in range(n_cycles):
        t_start = i * cycle_length
        t = rng.uniform(0, cycle_length, spots_per_cycle)
        t_sorted = np.sort(t)

        # Latitude: starts high (~30-35), migrates to low (~5-10)
        lat_mean = 35 - 25 * (t_sorted / cycle_length)
        lat_spread = 5 * (1 - 0.5 * t_sorted / cycle_length)
        lat = rng.normal(lat_mean, lat_spread)
        lat = np.clip(lat, 2, 45)

        # Hemisphere: random N or S
        hemisphere = rng.choice([-1, 1], spots_per_cycle)
        lat_signed = lat * hemisphere

        # Polarity: Hale's law
        # Even cycles (0, 2, ...): N hemisphere preceding = R (positive)
        # Odd cycles (1, 3, ...): N hemisphere preceding = V (negative)
        if i % 2 == 0:
            polarity = hemisphere  # N→+1 (R), S→-1 (V) for preceding
        else:
            polarity = -hemisphere  # N→-1 (V), S→+1 (R) for preceding

        # Add ~2.4% irregular polarity (Hale's statistic)
        n_irregular = int(0.024 * spots_per_cycle)
        irregular_idx = rng.choice(spots_per_cycle, n_irregular, replace=False)
        polarity[irregular_idx] *= -1

        all_time.append(t_sorted + t_start)
        all_lat.append(lat_signed)
        all_pol.append(polarity)

    return {
        'time': np.concatenate(all_time),
        'latitude': np.concatenate(all_lat),
        'polarity': np.concatenate(all_pol),
    }


# Generate data matching Hale's observation period (1903-1936 for context)
data = generate_butterfly_with_polarity(n_cycles=4, spots_per_cycle=400, seed=42)

# Map time to actual years (starting ~1903)
year_offset = 1903
time_years = data['time'] + year_offset

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True,
                                 gridspec_kw={'height_ratios': [3, 1]})

# --- Top: Butterfly diagram with polarity ---
pos_mask = data['polarity'] > 0
neg_mask = data['polarity'] < 0

ax1.scatter(time_years[pos_mask], data['latitude'][pos_mask],
            c='red', s=8, alpha=0.5, label='R (north-seeking, +)')
ax1.scatter(time_years[neg_mask], data['latitude'][neg_mask],
            c='blue', s=8, alpha=0.5, label='V (south-seeking, −)')

# Mark cycle boundaries (minima)
for yr in [1903, 1913.5, 1923.5, 1933.5]:
    ax1.axvline(yr, color='gray', ls='--', lw=1, alpha=0.5)

# Mark Hale's observation period
ax1.axvspan(1908, 1925, alpha=0.08, color='green', label='Hale observation period')

ax1.set_ylabel('Heliographic Latitude (°)', fontsize=13)
ax1.set_title("Synthetic Butterfly Diagram with Hale's Polarity Law\n"
              "합성 나비 다이어그램 + Hale 극성 법칙 (Fig. 18 재현)", fontsize=14)
ax1.legend(fontsize=10, loc='upper right', ncol=2)
ax1.set_ylim(-45, 45)
ax1.axhline(0, color='black', lw=1.5)

# Cycle labels
for i, (yr, label) in enumerate([(1906, 'Cycle 14\n(1st)'),
                                   (1917, 'Cycle 15\n(2nd)'),
                                   (1928, 'Cycle 16\n(3rd)')]):
    ax1.text(yr, 42, label, ha='center', fontsize=10,
             bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.8))

# Polarity annotations
ax1.annotate('N.Hem: R precedes\nS.Hem: V precedes',
             xy=(1917, -35), fontsize=9, ha='center',
             bbox=dict(boxstyle='round', fc='lightyellow'))
ax1.annotate('N.Hem: V precedes\nS.Hem: R precedes\n(REVERSED!)',
             xy=(1928, -35), fontsize=9, ha='center', color='darkred',
             bbox=dict(boxstyle='round', fc='mistyrose'))

# --- Bottom: Sunspot number proxy ---
t_fine = np.linspace(1903, 1936, 500)
# Simple sinusoidal proxy for sunspot number
ssn = 100 * np.abs(np.sin(np.pi * (t_fine - 1903) / 11.0))**1.5
ssn += 10 * np.random.default_rng(0).normal(size=len(t_fine))
ssn = np.clip(ssn, 0, None)

ax2.fill_between(t_fine, 0, ssn, alpha=0.3, color='orange')
ax2.plot(t_fine, ssn, 'k-', lw=0.8, alpha=0.5)
for yr in [1903, 1913.5, 1923.5, 1933.5]:
    ax2.axvline(yr, color='gray', ls='--', lw=1, alpha=0.5)

ax2.set_xlabel('Year', fontsize=13)
ax2.set_ylabel('Sunspot\nNumber', fontsize=11)
ax2.set_xlim(1903, 1936)

plt.tight_layout()
plt.show()

print("Key observation: Red (R) and Blue (V) swap between successive cycles!")
print("This polarity reversal defines the 22-year Hale cycle.")

## Part 2: Hale's Polarity Law — Schematic Diagram
## 파트 2: Hale 극성 법칙 — 도식적 다이어그램

극성 법칙의 핵심을 한눈에 보여주는 도식: 4개의 연속 11년 반주기에 걸친 선행/후행 흑점의 극성 패턴.

Schematic showing the core of the polarity law: preceding/following spot polarity patterns
across 4 consecutive 11-year half-cycles.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-2.5, 2.5)
ax.set_aspect('equal')
ax.axis('off')

cycle_labels = ['Cycle N\n(e.g. 14)', 'Cycle N+1\n(e.g. 15)',
                'Cycle N+2\n(e.g. 16)', 'Cycle N+3\n(e.g. 17)']
# Polarity patterns: (N_preceding, N_following, S_preceding, S_following)
# +1 = R (red/positive), -1 = V (blue/negative)
patterns = [
    (+1, -1, -1, +1),  # Cycle N
    (-1, +1, +1, -1),  # Cycle N+1 (reversed)
    (+1, -1, -1, +1),  # Cycle N+2 (same as N)
    (-1, +1, +1, -1),  # Cycle N+3 (same as N+1)
]

for i, (label, pattern) in enumerate(zip(cycle_labels, patterns)):
    x = i * 1.1
    # Column header
    ax.text(x, 2.2, label, ha='center', fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', fc='lightyellow', ec='gray'))

    for j, (hem_label, y_center) in enumerate([('North\n(N. Hem)', 1.0),
                                                 ('South\n(S. Hem)', -1.0)]):
        if i == 0:
            ax.text(-0.45, y_center, hem_label, ha='center', va='center',
                    fontsize=11, fontweight='bold')

        # Preceding and following spots
        p_pol = pattern[j * 2]      # preceding
        f_pol = pattern[j * 2 + 1]  # following

        # Draw spots (preceding = left, following = right)
        for k, (dx, pol, spot_label) in enumerate([(-0.15, p_pol, 'P'),
                                                     (0.15, f_pol, 'F')]):
            color = 'red' if pol > 0 else 'blue'
            fc = '#ff6666' if pol > 0 else '#6666ff'
            label_txt = 'R' if pol > 0 else 'V'
            circle = Circle((x + dx, y_center), 0.09, fc=fc, ec='black', lw=1.5)
            ax.add_patch(circle)
            ax.text(x + dx, y_center, label_txt, ha='center', va='center',
                    fontsize=10, fontweight='bold', color='white')
            ax.text(x + dx, y_center + 0.15, spot_label, ha='center',
                    fontsize=7, color='gray')

        # Arrow: rotation direction
        ax.annotate('', xy=(x + 0.25, y_center - 0.25),
                    xytext=(x - 0.25, y_center - 0.25),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1))
        ax.text(x, y_center - 0.35, '→ rotation', ha='center',
                fontsize=7, color='gray')

    # Reversal arrows between cycles
    if i < 3:
        ax.annotate('', xy=(x + 0.75, 0), xytext=(x + 0.35, 0),
                    arrowprops=dict(arrowstyle='->', color='green', lw=2))

# Bracket for 22-year cycle
ax.annotate('', xy=(0, -2.1), xytext=(2.2, -2.1),
            arrowprops=dict(arrowstyle='<->', color='darkred', lw=2.5))
ax.text(1.1, -2.3, '22-year Hale Cycle\n(same polarity repeats)',
        ha='center', fontsize=12, color='darkred', fontweight='bold')

# Bracket for 11-year half-cycle
ax.annotate('', xy=(0, -1.7), xytext=(1.1, -1.7),
            arrowprops=dict(arrowstyle='<->', color='orange', lw=2))
ax.text(0.55, -1.85, '11-yr', ha='center', fontsize=10, color='orange')

# Equator line
ax.axhline(0, xmin=0.05, xmax=0.95, color='black', lw=2, ls='-')
ax.text(4.6, 0, 'Equator\n(적도)', ha='center', va='center', fontsize=10)

ax.set_title("Hale's Polarity Law — Schematic\nHale 극성 법칙 도식\n"
             "(P = Preceding, F = Following; R = North-seeking/+, V = South-seeking/−)",
             fontsize=14)
plt.tight_layout()
plt.show()

print("97.6% of 1,735 observed groups followed this exact pattern!")
print("Only 2.4% (41 groups) were exceptions.")

## Part 3: Classification Statistics — Reproducing Tables I–II
## 파트 3: 분류 통계 — Tables I–II 재현

Hale & Nicholson의 Table I (2,174개 흑점군 분류)과 Table II (연도별 %)를 시각화합니다.

Visualizing Table I (classification of 2,174 groups) and Table II (yearly percentages).

In [ ]:
# Table I data: Magnetic Classification of 2174 Spot Groups
table1_years = list(range(1915, 1925))
table1_data = {
    'α':  [17, 53, 45, 64, 52, 21, 11, 12, 1, 8],
    'αp': [33, 54, 85, 75, 54, 33, 34, 17, 7, 17],
    'αf': [9, 12, 15, 10, 11, 9, 5, 4, 2, 3],
    'β':  [38, 66, 88, 85, 47, 30, 26, 17, 5, 21],
    'βp': [51, 100, 134, 101, 77, 46, 28, 18, 16, 21],
    'βf': [9, 26, 38, 33, 22, 12, 8, 3, 2, 6],
    'βγ': [8, 13, 15, 8, 9, 4, 2, 0, 0, 2],
    'γ':  [3, 0, 2, 1, 3, 4, 1, 2, 0, 1],
}

# Table II data: Percentage of Spot Groups in Each Class (mean 1915-1924)
mean_pct = {'α': 14, 'αp': 20, 'αf': 4, 'β': 21, 'βp': 29, 'βf': 8, 'βγ': 3, 'γ': 1}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Pie chart of mean classification
classes = list(mean_pct.keys())
values = list(mean_pct.values())
colors_pie = ['#ff9999', '#ffcc99', '#ffe6cc',
              '#99ccff', '#6699ff', '#99e6ff', '#cce6ff',
              '#d9d9d9']
explode = [0, 0, 0, 0.05, 0.05, 0, 0, 0]

wedges, texts, autotexts = axes[0].pie(values, labels=classes, autopct='%1.0f%%',
                                         colors=colors_pie, explode=explode,
                                         startangle=90, pctdistance=0.8)
for t in autotexts:
    t.set_fontsize(10)

# Add unipolar/bipolar grouping
axes[0].text(-1.4, -1.3, f'Unipolar (α): {14+20+4}%', fontsize=11,
             color='red', fontweight='bold')
axes[0].text(-1.4, -1.5, f'Bipolar (β): {21+29+8+3}%', fontsize=11,
             color='blue', fontweight='bold')
axes[0].text(-1.4, -1.7, f'Complex (γ): 1%', fontsize=11, color='gray')

axes[0].set_title('Mean Magnetic Classification (1915–1924)\n'
                   '평균 자기 분류 (Table II)', fontsize=13)

# Right: Stacked bar chart by year
bottom = np.zeros(len(table1_years))
bar_colors = {'α': '#ff9999', 'αp': '#ffcc99', 'αf': '#ffe6cc',
              'β': '#99ccff', 'βp': '#6699ff', 'βf': '#99e6ff',
              'βγ': '#cce6ff', 'γ': '#d9d9d9'}

for cls in ['α', 'αp', 'αf', 'β', 'βp', 'βf', 'βγ', 'γ']:
    vals = np.array(table1_data[cls])
    axes[1].bar(table1_years, vals, bottom=bottom, label=cls,
                color=bar_colors[cls], edgecolor='white', width=0.8)
    bottom += vals

axes[1].set_xlabel('Year', fontsize=12)
axes[1].set_ylabel('Number of Spot Groups', fontsize=12)
axes[1].set_title('Magnetic Classification by Year (Table I)\n'
                   '연도별 자기 분류 (Table I)', fontsize=13)
axes[1].legend(fontsize=9, ncol=4, loc='upper right')
axes[1].set_xticks(table1_years)
axes[1].set_xticklabels(table1_years, rotation=45)

plt.tight_layout()
plt.show()

# Print summary
total = sum(sum(v) for v in table1_data.values())
uni = sum(table1_data['α']) + sum(table1_data['αp']) + sum(table1_data['αf'])
bip = sum(table1_data['β']) + sum(table1_data['βp']) + sum(table1_data['βf']) + sum(table1_data['βγ'])
print(f"Total classified groups: {total}")
print(f"Unipolar (α family): {uni} ({100*uni/total:.0f}%)")
print(f"Bipolar (β family):  {bip} ({100*bip/total:.0f}%)")
print(f"~90% are or evolve into bipolar — the basis for the polarity law")

## Part 4: Bipolar Group as Flux Tube — Modern Interpretation
## 파트 4: 쌍극군의 자기 flux tube 구조 — 현대적 해석

Hale의 관측을 현대적으로 해석하면, 쌍극 흑점군은 태양 내부에서 부상한 Ω-형 자기 flux tube의 양쪽 발(footpoint)입니다.
Hale의 극성 법칙은 이 flux tube가 toroidal 자기장에서 기원함을 반영합니다.

In modern interpretation, bipolar groups are the two footpoints of an Ω-shaped magnetic flux tube
risen from the solar interior. Hale's polarity law reflects these tubes originating from the toroidal field.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax_idx, (ax, cycle_name, toroidal_dir) in enumerate(zip(
    axes, ['Cycle N (e.g. 15)', 'Cycle N+1 (e.g. 16)'], [1, -1])):

    ax.set_xlim(-3, 3)
    ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal')
    ax.grid(False)
    ax.set_facecolor('#fffdf0')

    # Solar surface
    ax.axhline(0, color='orange', lw=3, label='Photosphere (광구)')
    ax.fill_between([-3, 3], [-2.5, -2.5], [0, 0], alpha=0.1, color='orange')
    ax.text(-2.8, -0.3, 'Interior\n(내부)', fontsize=9, color='orange')
    ax.text(-2.8, 0.3, 'Atmosphere\n(대기)', fontsize=9, color='gray')

    # Toroidal field direction (subsurface)
    arrow_color = 'blue' if toroidal_dir > 0 else 'red'
    for y_arr in [-1.5]:
        ax.annotate('', xy=(2.5 * toroidal_dir, y_arr),
                    xytext=(-2.5 * toroidal_dir, y_arr),
                    arrowprops=dict(arrowstyle='->', color=arrow_color,
                                    lw=3, alpha=0.3))
    ax.text(0, -2.0, f'Toroidal field {"→" if toroidal_dir > 0 else "←"}',
            ha='center', fontsize=10, color=arrow_color, alpha=0.5)

    # Omega-shaped flux tube
    t = np.linspace(-1.5, 1.5, 100)
    tube_y = -0.8 * t**2 + 1.8  # Parabolic arch
    tube_y_sub = np.where(tube_y < 0, tube_y, 0)

    # Draw tube (subsurface part)
    ax.plot(t, np.minimum(tube_y, 0), color='purple', lw=4, alpha=0.3)
    # Draw tube (above surface)
    above = tube_y > 0
    ax.plot(t[above], tube_y[above], color='purple', lw=4, label='Ω flux tube')

    # Footpoints (spots)
    # Preceding (west) = left, Following (east) = right
    if toroidal_dir > 0:
        # N hemisphere, Cycle N: preceding = R, following = V
        p_color, f_color = 'red', 'blue'
        p_label, f_label = 'R (+)', 'V (−)'
    else:
        # N hemisphere, Cycle N+1: preceding = V, following = R
        p_color, f_color = 'blue', 'red'
        p_label, f_label = 'V (−)', 'R (+)'

    # Preceding spot (left footpoint)
    ax.add_patch(Circle((-1.2, 0.15), 0.25, fc=p_color, ec='black', lw=2, alpha=0.7))
    ax.text(-1.2, 0.15, p_label, ha='center', va='center', fontsize=10,
            fontweight='bold', color='white')
    ax.text(-1.2, 0.55, 'Preceding\n(선행)', ha='center', fontsize=9)

    # Following spot (right footpoint)
    ax.add_patch(Circle((1.2, 0.15), 0.20, fc=f_color, ec='black', lw=2, alpha=0.7))
    ax.text(1.2, 0.15, f_label, ha='center', va='center', fontsize=9,
            fontweight='bold', color='white')
    ax.text(1.2, 0.55, 'Following\n(후행)', ha='center', fontsize=9)

    # Field lines through tube (arrows along arch)
    for frac in [0.3, 0.5, 0.7]:
        idx = int(frac * len(t))
        if above[idx]:
            dx = t[min(idx+1, len(t)-1)] - t[idx]
            dy = tube_y[min(idx+1, len(t)-1)] - tube_y[idx]
            ax.annotate('', xy=(t[idx] + 0.15*dx/max(abs(dx), 0.01),
                                tube_y[idx] + 0.15*dy/max(abs(dy), 0.01)),
                        xytext=(t[idx], tube_y[idx]),
                        arrowprops=dict(arrowstyle='->', color='purple', lw=1.5))

    # Rotation direction
    ax.annotate('← Solar rotation', xy=(-2.5, 2.2), fontsize=10, color='gray')

    ax.set_title(f'{cycle_name}\nNorthern Hemisphere (북반구)', fontsize=13)
    if ax_idx == 0:
        ax.legend(fontsize=9, loc='upper right')

plt.suptitle("Bipolar Sunspot Groups as Ω-Shaped Flux Tubes\n"
             "쌍극 흑점군 = Ω자형 자기 flux tube\n"
             "(Toroidal field reversal → Polarity reversal between cycles)",
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("Modern interpretation (Babcock 1961, Leighton 1969):")
print("- Differential rotation winds poloidal field into toroidal field")
print("- Toroidal field direction reverses each cycle → polarity reverses")
print("- Buoyant Ω-loops rise to surface → bipolar sunspot groups")
print("- Hale's law is a direct consequence of the toroidal field direction")

## Summary / 요약

| Part | 내용 / Content | Hale & Nicholson (1925) 연결 / Connection |
|---|---|---|
| 1 | Butterfly diagram + polarity 시뮬레이션 | Fig. 18 재현 — 22년 자기 주기의 시각적 증거 |
| 2 | 극성 법칙 도식 | 4개 연속 반주기에 걸친 극성 반전 패턴 — 2.4% 예외 |
| 3 | Tables I–II 분류 통계 시각화 | 2,174개 흑점군의 α/β/γ 분류 — 쌍극 지배적 |
| 4 | Ω-flux tube 모델 | 현대적 해석 — toroidal field 방향이 극성 결정 |

**다음 논문 / Next paper**: Alfvén (1942) — MHD 파동의 존재 예측.
Hale의 자기장 + Evershed의 흐름 + Hale의 극성 법칙이 모두 결합되면, 자기장과 플라즈마의
상호작용을 기술하는 **MHD 이론**이 필요합니다. Alfvén이 그 기초를 놓습니다.

**Next paper**: Alfvén (1942) — Prediction of MHD waves.
Hale's field + Evershed's flow + Hale's polarity law combined demand **MHD theory** to describe
magnetic field–plasma interactions. Alfvén lays that foundation.